In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
from PIL import Image
import torch.nn.functional as F
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

class SCINPatchDataset(Dataset):
    """
    A PyTorch Dataset implementation for the Hugging Face SCIN dataset.
    Extracts face skin patches dynamically using a semantic segmentation model.
    Implements lazy-loading of the Segformer model to prevent CUDA multiprocessing
    initialization errors (`Cannot re-initialize CUDA in forked subprocess`) when num_workers > 0.
    """
    def __init__(self, hf_dataset_name: str = "google/scin", split: str = "train",
                 transform=None, device=None):
        super().__init__()
        self.transform = transform
        # Dataloader threads shouldn't instantiate Models during __init__ on the main thread
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.device_type = 'cuda' if torch.cuda.is_available() else 'cpu'

        # 1. Load the SCIN dataset from Hugging Face
        raw_dataset = load_dataset(hf_dataset_name, split=split)

        # 2. Handle missing labels: Filter ungradable images and null targets
        self.dataset = raw_dataset.filter(
            lambda x: x.get('gradable_for_monk_skin_tone_us') is True and
                      x.get('monk_skin_tone_label_us') is not None
        )

        # Do NOT initialize the Segformer model here. It breaks num_workers > 0.
        self.seg_processor = None
        self.seg_model = None

    def _init_model_lazy(self):
        """Initializes the Segmenter on the specific isolated background thread."""
        if self.seg_processor is None or self.seg_model is None:
            self.seg_processor = SegformerImageProcessor.from_pretrained("jonathandinu/face-parsing")
            self.seg_model = SegformerForSemanticSegmentation.from_pretrained("jonathandinu/face-parsing")
            self.seg_model = self.seg_model.to(self.device).eval()

    def _extract_training_patch(self, image: Image.Image) -> Image.Image:
        """
        Runs the semantic segmentation model to isolate the 'skin' class.
        Extracts a bounding box around the largest skin region to serve as the training patch.
        """
        self._init_model_lazy()

        # Preprocess image for the Segformer
        inputs = self.seg_processor(images=image, return_tensors="pt").to(self.device)

        # Force torch.autocast to vastly accelerate CPU or GPU interference times
        # for the expensive transformer forward passes taking place inside a Dataloader loop
        with torch.inference_mode(), torch.autocast(device_type=self.device_type):
            outputs = self.seg_model(**inputs)
            logits = outputs.logits

        # Resize logits to match the input image spatial resolution (uses CPU for simplicity)
        # We've already completed the dense inference
        upsampled_logits = F.interpolate(
            logits.float(), # ensure float32 just in case of autocast
            size=image.size[::-1], # PIL size is (W, H); interpolate requires (H, W)
            mode='bilinear',
            align_corners=False
        )

        # The skin class index for jonathandinu/face-parsing is 1
        semantic_map = upsampled_logits.argmax(dim=1).squeeze().cpu().numpy()
        skin_mask = (semantic_map == 1).astype(np.uint8)

        # Extract patch based on the mask
        if np.sum(skin_mask) > 0:
            y_indices, x_indices = np.where(skin_mask == 1)
            y_min, y_max = y_indices.min(), y_indices.max()
            x_min, x_max = x_indices.min(), x_indices.max()

            # Add a small buffer to the bounding box if possible
            h, w = skin_mask.shape
            y_min, y_max = max(0, y_min - 10), min(h, y_max + 10)
            x_min, x_max = max(0, x_min - 10), min(w, x_max + 10)

            patch = image.crop((x_min, y_min, x_max, y_max))
            return patch
        else:
            # Fallback to the original image if no skin is confidently detected
            return image

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, idx: int):
        item = self.dataset[idx]

        # Extract the raw PIL Image
        keys_to_try = ['image', 'image_1', 'image_1_path', 'image_2_path', 'image_path']
        image_obj = None
        for k in keys_to_try:
            if k in item and item[k] is not None:
                image_obj = item[k]
                break
        if image_obj is None:
            # Fallback to white patch to not crash dataloader on ungradable ones
            image_obj = Image.new('RGB', (224, 224), color='white')

        image = image_obj
        if not isinstance(image, Image.Image):
            image = Image.open(image)
        image = image.convert("RGB")

        # Dynamically extract the skin patch using the face parser
        skin_patch = self._extract_training_patch(image)

        # Extract the Monk Skin Tone Label (1 to 10)
        # Shift to 0-indexed tensor (0 to 9) for standard PyTorch Loss criteria
        label_val = float(item['monk_skin_tone_label_us'])
        label = torch.tensor(int(label_val) - 1, dtype=torch.long)

        # Apply standard transform pipelines (resize, normalization, augmentation)
        if self.transform:
            image_tensor = self.transform(skin_patch)
        else:
            image_tensor = transforms.ToTensor()(skin_patch)

        return image_tensor, label

def get_scin_dataloader(batch_size: int = 32, num_workers: int = 0):
    """
    Constructs and returns the training DataLoader.
    Applies spatial augmentations while preserving photometric skin integrity.
    """
    # Define standard transform pipelines
    train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),  # convert PIL image to Tensor
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # typical ImageNet means
        std=[0.229, 0.224, 0.225]    # your standard deviations
    )
])


    # Initialize the dynamically extracting dataset
    train_dataset = SCINPatchDataset(split="train", transform=train_transform)

    # Configure the DataLoader with memory pinning for optimal GPU transfer
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        multiprocessing_context="spawn" if num_workers > 0 else None,
        drop_last=True
    )

    return train_loader


In [ ]:
import os
import time
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet50, ResNet50_Weights
# from torch.cuda.amp import GradScaler, autocast
from torch.amp import GradScaler


def build_skin_tone_classifier(num_classes: int = 10) -> nn.Module:
    """
    Instantiates a pre-trained ResNet50 model and modifies the terminal
    fully connected layer for the specified number of target classes.
    """
    # Load the most accurate available ImageNet pre-trained weights
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

    # Replace the classification head to map to the 10 MST categories
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    return model

def save_checkpoint(state: dict, filename: str):
    """Serializes the comprehensive model and optimizer state to disk."""
    print(f"=> Saving checkpoint statistics at {filename}")
    torch.save(state, filename)

def train_skin_tone_model(
    model: nn.Module,
    train_loader: DataLoader,
    num_epochs: int = 25,
    learning_rate: float = 1e-4,
    checkpoint_dir: str = "./checkpoints"
):
    """
    Production-grade CNN training loop executing empirical risk minimization.
    Implements Automatic Mixed Precision (AMP) and comprehensive state checkpointing.
    """
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Auto-detect hardware accelerator (CUDA for NVIDIA, MPS for Apple, or CPU)
    device = torch.device(
        "cuda" if torch.cuda.is_available() else
        "mps" if torch.backends.mps.is_available() else "cpu"
    )
    model = model.to(device)

    # Define the objective function and the optimization algorithm
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)

    """
    # Initialize the AMP GradScaler for float16 memory efficiency
    # scaler = GradScaler()
    """

    # Initialize the AMP GradScaler for float16 memory efficiency
    device_type = 'cuda' if torch.cuda.is_available() else 'cpu'
    scaler = torch.amp.GradScaler(device_type)


    best_loss = float('inf')

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        start_time = time.time()

        for batch_idx, (images, labels) in enumerate(train_loader):
            # Pin memory transfers by setting non_blocking=True
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            # Clear gradients efficiently
            optimizer.zero_grad(set_to_none=True)

            # Execute Forward pass within the Mixed Precision context
            # with autocast():
            #     outputs = model(images)
            #     loss = criterion(outputs, labels)

            # Execute Forward pass within the Mixed Precision context
            with torch.autocast(device_type=device_type):
                outputs = model(images)
                loss = criterion(outputs, labels)


            # Execute Backward pass scaling to prevent float16 underflow
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            # Aggregate Batch Statistics
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct_predictions += (predicted == labels).sum().item()
            total_samples += labels.size(0)

        epoch_loss = running_loss / total_samples
        epoch_acc = correct_predictions / total_samples
        epoch_time = time.time() - start_time

        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Time: {epoch_time:.2f}s | "
              f"Loss: {epoch_loss:.4f} | "
              f"Accuracy: {epoch_acc*100:.2f}%")

        # Construct the comprehensive checkpoint state dictionary
        checkpoint_state = {
            'epoch': epoch + 1,
            'state_dict': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scaler': scaler.state_dict(),
            'loss': epoch_loss,
            'accuracy': epoch_acc
        }

        # Save the latest epoch unconditionally
        latest_path = os.path.join(checkpoint_dir, "latest_checkpoint.pth.tar")
        save_checkpoint(checkpoint_state, latest_path)

        # Conditionally save the best performing model
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_path = os.path.join(checkpoint_dir, "best_model.pth.tar")
            save_checkpoint(checkpoint_state, best_path)

    return model

In [ ]:
from typing import List

def extract_skin_patches(
    image: Image.Image,
    mask: np.ndarray,
    patch_size: int = 224,
    stride: int = 112,
    purity_threshold: float = 0.85
) -> List[Image.Image]:
    """
    Traverses an image and its corresponding segmentation mask to extract
    localized patches that predominantly consist of valid skin pixels.

    Args:
        image (PIL.Image): The original input RGB image.
        mask (np.ndarray): Binary segmentation mask (1 for skin, 0 otherwise).
        patch_size (int): Spatial dimension of the extracted square patch.
        stride (int): Step size for the sliding window traversal algorithm.
        purity_threshold (float): Minimum required fraction of skin pixels in the patch.

    Returns:
        List[PIL.Image]: A collection of valid, cropped skin patches ready for classification.
    """
    img_array = np.array(image)
    h, w, _ = img_array.shape
    patches = []

    # Sliding window extraction across the image height and width
    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            # Extract the local sub-patch from the binary mask
            mask_patch = mask[y:y+patch_size, x:x+patch_size]

            # Calculate the density of skin pixels within this specific spatial window
            skin_ratio = np.sum(mask_patch) / (patch_size * patch_size)

            # Accept the image patch if it exceeds the strict purity constraint
            if skin_ratio >= purity_threshold:
                img_patch = img_array[y:y+patch_size, x:x+patch_size, :]
                patches.append(Image.fromarray(img_patch))

    # Algorithmic Fallback Mechanism:
    # If no patch meets the strict 85% purity (e.g., small faces or low resolution),
    # extract the global bounding box of the largest connected skin component.
    if not patches and np.sum(mask) > 0:
        y_indices, x_indices = np.where(mask == 1)
        y_min, y_max = y_indices.min(), y_indices.max()
        x_min, x_max = x_indices.min(), x_indices.max()

        # Crop to the global bounding box and forcefully resize to the required patch_size
        fallback_patch = image.crop((x_min, y_min, x_max, y_max))
        fallback_patch = fallback_patch.resize((patch_size, patch_size), Image.BILINEAR)
        patches.append(fallback_patch)

    return patches

In [ ]:
from typing import Tuple
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

def predict_patches(
    patches: List[Image.Image],
    model: nn.Module,
    device: torch.device
) -> Tuple[int, float]:
    """
    Executes batched inference over a collection of image patches, applying
    softmax and mean pooling to derive a consensus skin tone prediction.
    """
    if not patches:
        raise ValueError("No valid skin patches were extracted from the input image.")

    # Standard deterministic inference transforms matching the training state
    inference_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),  # convert PIL image to Tensor
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],  # typical ImageNet means
            std=[0.229, 0.224, 0.225]    # your standard deviations
        )
    ])


    # Process patches into a single batched tensor and move to the target hardware device
    tensor_batch = torch.stack([inference_transform(p) for p in patches]).to(device)

    # Ensure Dropout and BatchNorm layers are locked in evaluation mode
    model.eval()

    with torch.inference_mode():
        logits = model(tensor_batch)

        # Convert raw logits to probability distributions via the Softmax activation
        probabilities = torch.nn.functional.softmax(logits, dim=1)

        # Aggregate (Mean Pool) probabilities across all patches for the given face
        consensus_probs = probabilities.mean(dim=0)

        # Extract the maximum probabilistic confidence and its categorical index
        confidence_score, predicted_idx = torch.max(consensus_probs, dim=0)

    # Shift 0-indexed tensor output back to the standard 1-indexed Monk Skin Tone scale
    predicted_mst = predicted_idx.item() + 1
    confidence_percentage = confidence_score.item() * 100.0

    return predicted_mst, confidence_percentage

def predict_skin_tone(image_path: str) -> Tuple[str, float]:
    """
    Top-level API: Reads an image, extracts face skin regions via semantic
    segmentation, runs the ResNet classifier on local patches, and aggregates
    predictions to return the final Monk Skin Tone label and confidence percentage.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else
                          "mps" if torch.backends.mps.is_available() else "cpu")

    # 1. Load the target image from the local filesystem
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        raise IOError(f"Failed to load image at {image_path}: {e}")

    # 2. Initialize the Semantic Segmentation (Face Parsing) framework
    model_name = "jonathandinu/face-parsing"
    seg_processor = SegformerImageProcessor.from_pretrained(model_name)
    seg_model = SegformerForSemanticSegmentation.from_pretrained(model_name).to(device)
    seg_model.eval()

    # 3. Generate pure binary skin mask utilizing the model
    inputs = seg_processor(images=image, return_tensors="pt").to(device)
    with torch.inference_mode():
        seg_outputs = seg_model(**inputs)

    upsampled_logits = F.interpolate(
        seg_outputs.logits, size=image.size[::-1], mode='bilinear', align_corners=False
    )
    semantic_map = upsampled_logits.argmax(dim=1).squeeze().cpu().numpy()
    skin_mask = (semantic_map == 1).astype(np.uint8)  # Index 1 is the 'skin' class

    # 4. Extract localized skin patches using the sliding window methodology
    patches = extract_skin_patches(image, skin_mask)

    # 5. Initialize the trained ResNet classifier
    classifier = build_skin_tone_classifier(num_classes=10)
    checkpoint_path = "./checkpoints/best_model.pth.tar"

    if os.path.exists(checkpoint_path):
        # Load the saved state dictionary generated by the training loop
        checkpoint = torch.load(checkpoint_path, map_location=device)
        classifier.load_state_dict(checkpoint['state_dict'])
    else:
        print("Warning: Utilizing untrained weights. Valid checkpoint not found.")

    classifier.to(device)

    # 6. Aggregate predictions across all extracted patches
    mst_class, confidence = predict_patches(patches, classifier, device)

    # Format the final output string
    label = f"Monk Skin Tone (MST) {mst_class}"
    return label, confidence


In [ ]:
# --- Training Execution Cell ---
# 1. Initialize the DataLoader. (Reduced batch_size and num_workers for local running/demonstration)
print('Initializing DataLoader...')
train_loader = get_scin_dataloader(batch_size=8, num_workers=0)

# 2. Build the ResNet classifier
print('Building Skin Tone Classifier...')
classifier = build_skin_tone_classifier(num_classes=10)

# 3. Execute the training loop
# Note: num_epochs is set to 5 for demonstration.
print('Starting Training Loop...')
trained_model = train_skin_tone_model(
    model=classifier,
    train_loader=train_loader,
    num_epochs=25,
    learning_rate=1e-4,
    checkpoint_dir='./checkpoints'
)
print('Training Complete. Best model saved to ./checkpoints/best_model.pth.tar')

Initializing DataLoader...


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Some datasets params were ignored: ['splits', 'download_size', 'dataset_size']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

data/train-00001-of-00026.parquet:   0%|          | 0.00/476M [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/374 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/339M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1172 [00:00<?, ?it/s]

Epoch [1/25] Time: 885.99s | Loss: 1.5930 | Accuracy: 37.04%
=> Saving checkpoint statistics at ./checkpoints/latest_checkpoint.pth.tar
=> Saving checkpoint statistics at ./checkpoints/best_model.pth.tar
Epoch [2/25] Time: 864.69s | Loss: 1.4148 | Accuracy: 42.56%
=> Saving checkpoint statistics at ./checkpoints/latest_checkpoint.pth.tar
=> Saving checkpoint statistics at ./checkpoints/best_model.pth.tar
Epoch [3/25] Time: 861.38s | Loss: 1.3020 | Accuracy: 47.64%
=> Saving checkpoint statistics at ./checkpoints/latest_checkpoint.pth.tar
=> Saving checkpoint statistics at ./checkpoints/best_model.pth.tar
Epoch [4/25] Time: 858.57s | Loss: 1.1552 | Accuracy: 55.04%
=> Saving checkpoint statistics at ./checkpoints/latest_checkpoint.pth.tar
=> Saving checkpoint statistics at ./checkpoints/best_model.pth.tar
Epoch [5/25] Time: 859.69s | Loss: 0.9316 | Accuracy: 65.20%
=> Saving checkpoint statistics at ./checkpoints/latest_checkpoint.pth.tar
=> Saving checkpoint statistics at ./checkpoints

In [ ]:
img_path = "/content/custom_image1.jpg"
label, confidence = predict_skin_tone(img_path)
print(f"Predicted Label: {label}")
print(f"Confidence: {confidence:.2f}%")